# Mais exemplos de geradores

Este notebook apresenta formas mais avancadas de usar geradores em Python.
Os exemplos mostram que geradores nao servem apenas para percorrer listas:
eles tambem podem criar sequencias sob demanda, manter estado interno,
receber valores externos e compor pipelines de processamento.

Ao longo dos exemplos, aparecem cinco usos comuns:

- expressao geradora para criar valores sem montar uma lista completa;
- gerador infinito controlado por `next()`;
- comunicacao com o gerador usando `send()`;
- tratamento de excecoes com `throw()`;
- encadeamento de geradores para transformar dados em etapas.

In [1]:
# Exemplo 1 - Expressao geradora.
# Com parenteses, o Python cria um gerador em vez de uma lista.
# Isso significa que os quadrados sao calculados sob demanda, item por item.
# Se fosse [x ** 2 for x in range(10)], todos os valores seriam calculados
# imediatamente e armazenados na memoria como uma lista.
quadrados = (x ** 2 for x in range(10))

for quadrado in quadrados:
    print(quadrado)


# Resultado esperado:
# 0  -> quadrado de 0.
# 1  -> quadrado de 1.
# 4  -> quadrado de 2.
# 9  -> quadrado de 3.
# 16 -> quadrado de 4.
# 25 -> quadrado de 5.
# 36 -> quadrado de 6.
# 49 -> quadrado de 7.
# 64 -> quadrado de 8.
# 81 -> quadrado de 9.


0
1
4
9
16
25
36
49
64
81


In [3]:
# Exemplo 2 - Gerador infinito.
# O gerador nunca termina sozinho, por isso o consumo precisa ser controlado.
# No laco `for _ in range(10)`, o `_` indica que o valor da repeticao
# sera ignorado; ele serve apenas para repetir a chamada 10 vezes.
def contador_infinito():
    """Gera numeros inteiros em sequencia crescente, sem limite."""

    n = 0
    while True:
        yield n
        n += 1

contador = contador_infinito()
for _ in range(10):
    print(next(contador))


# Resultado esperado:
# 0 -> primeiro valor gerado.
# 1 -> segunda chamada de `next(contador)`.
# 2 -> o contador segue incrementando.
# 3 -> mais um valor da sequencia.
# 4 -> continuacao da contagem.
# 5 -> metade das 10 leituras feitas.
# 6 -> novo numero inteiro consecutivo.
# 7 -> sequencia continua sem pular valores.
# 8 -> penultimo valor impresso neste exemplo.
# 9 -> decima leitura; o loop para aqui por causa do `range(10)`.


0
1
2
3
4
5
6
7
8
9


In [1]:
# Exemplo 3 - Envio de valores com `send()`.
# O gerador devolve o total atual e pode receber novos valores para acumular.
def gerador_soma():
    """Mantem um acumulador interno atualizado por `send()`."""

    total = 0
    while True:
        valor = yield total
        if valor is not None:
            total += valor

soma = gerador_soma()
# `next()` inicia o gerador e avanca ate o primeiro `yield`.
# Aqui ele so devolve o total atual (0), sem enviar um valor para `valor`.
print(next(soma))
# `send(10)` retoma o gerador, envia 10 para `valor` e devolve o novo total.
print(soma.send(10))
# `send(5)` envia outro valor para acumular e devolve o total atualizado.
print(soma.send(5))


# Resultado esperado:
# 0  -> valor inicial do acumulador antes de qualquer soma.
# 10 -> total depois de receber `10` via `send(10)`.
# 15 -> total depois de somar mais `5` ao acumulado anterior.


0
10
15


In [5]:
# Exemplo 4 - Tratamento de excecoes com `throw()`.
# Uma excecao pode ser lancada para dentro do gerador e tratada internamente.
def gerador_excecao():
    """Demonstra como um gerador pode reagir a excecoes externas."""

    try:
        yield 'Gerador iniciado'
        yield 'Gerador em execucao'
    except Exception as e:
        yield f'Excecao capturada: {e}'

gerador = gerador_excecao()
print(next(gerador))
print(next(gerador))
print(gerador.throw(ValueError('Erro de valor')))


# Resultado esperado:
# Gerador iniciado                    -> primeira mensagem produzida pelo gerador.
# Gerador em execucao                 -> segunda mensagem antes da excecao.
# Excecao capturada: Erro de valor    -> resposta do `except` apos o `throw()`.


Gerador iniciado
Gerador em execucao
Excecao capturada: Erro de valor


In [ ]:
# Exemplo 4.1 - Tratamento de excecoes sem `throw()`.
# A excecao acontece dentro do proprio gerador e e tratada no `try/except`.
def gerador_divisao(valores):
    """Divide 10 por cada valor e trata divisao por zero."""

    for valor in valores:
        try:
            yield 10 / valor
        except ZeroDivisionError:
            yield 'Nao e possivel dividir por zero'

for resultado in gerador_divisao([2, 1, 0, 5]):
    print(resultado)


# Resultado esperado:
# 5.0                           -> 10 / 2.
# 10.0                          -> 10 / 1.
# Nao e possivel dividir por zero -> tentativa de 10 / 0 tratada pelo `except`.
# 2.0                           -> 10 / 5 apos continuar a iteracao.


In [6]:
# Exemplo 5 - Cascata de geradores.
# Cada gerador transforma os dados e passa o resultado para a proxima etapa.
def multiplicar_por_dois(iterable):
    """Multiplica cada item por 2."""

    for item in iterable:
        yield item * 2

def adicionar_cinco(iterable):
    """Soma 5 a cada item recebido."""

    for item in iterable:
        yield item + 5

numeros = range(5)
resultado = adicionar_cinco(multiplicar_por_dois(numeros))

for valor in resultado:
    print(valor)


# Resultado esperado:
# 5  -> (0 * 2) + 5.
# 7  -> (1 * 2) + 5.
# 9  -> (2 * 2) + 5.
# 11 -> (3 * 2) + 5.
# 13 -> (4 * 2) + 5.


5
7
9
11
13
